# GraphoVision — Colab 학습 & 시각화

### 실행 전 준비사항
1. **런타임 → 런타임 유형 변경 → T4 GPU** 선택
2. 아래 셀을 **순서대로** 실행

### 데이터 업로드 방법 (두 가지 중 택일)
- **방법 A (추천):** Google Drive에 `Graphovision` 폴더 통째로 업로드 → 셀 1A 사용
- **방법 B:** Colab에 직접 ZIP 업로드 → 셀 1B 사용

## 셀 1A. Google Drive 마운트 (추천)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# ★ 본인 Drive 경로에 맞게 수정
PROJECT_DIR = '/content/drive/MyDrive/Graphovision'
os.chdir(PROJECT_DIR)
print('작업 디렉토리:', os.getcwd())
print('파일 목록:', os.listdir('.'))

## 셀 1B. ZIP 직접 업로드 (Drive 없을 때)

In [ ]:
# 셀 1A를 실행했으면 이 셀은 건너뛰세요
from google.colab import files
import zipfile, os

uploaded = files.upload()  # Graphovision.zip 업로드
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

PROJECT_DIR = '/content/Graphovision'
os.chdir(PROJECT_DIR)
print('작업 디렉토리:', os.getcwd())

## 셀 2. 패키지 설치 및 GPU 확인

In [ ]:
!pip install -q torch torchvision scikit-learn pandas matplotlib Pillow

import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU 메모리:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 셀 3. 데이터 파이프라인 검증

In [ ]:
import sys
sys.path.insert(0, '.')

from data_pipeline import parse_label_list, build_writer_form_map, build_sample_list, get_dataloaders
import numpy as np

label_df        = parse_label_list('label_list.txt')
writer_form_map = build_writer_form_map('xml')
samples         = build_sample_list(label_df, writer_form_map, 'lines')

print('\n=== 레이블 분포 (1의 비율) ===')
labels_all = np.array([s['labels'] for s in samples])
for i in range(8):
    ratio = labels_all[:, i].mean()
    print(f'  label_{i}: {ratio:.3f} ({int(labels_all[:,i].sum())} / {len(samples)})')

## 셀 4. 학습 (GPU 풀학습)

In [ ]:
import json, time
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from pathlib import Path

from data_pipeline import get_dataloaders
from model import GraphoVisionCNN

# ── 설정 ──────────────────────────────────────
BATCH_SIZE = 64      # GPU이므로 배치 크기 확대
EPOCHS     = 50
LR         = 1e-3
DROPOUT    = 0.3
PATIENCE   = 10
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE       = Path('.')
print(f'Device: {DEVICE}')

# ── 데이터 로드 ────────────────────────────────
train_loader, val_loader, test_loader = get_dataloaders(
    str(BASE/'lines'), str(BASE/'xml'), str(BASE/'label_list.txt'),
    batch_size=BATCH_SIZE, num_workers=2
)

# ── 모델 & Loss ────────────────────────────────
model = GraphoVisionCNN(dropout=DROPOUT).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'모델 파라미터: {total_params:,}개 ({total_params/1e6:.2f}M)')

all_labels = torch.cat([l for _, l in train_loader])
pos_count  = all_labels.sum(dim=0)
neg_count  = all_labels.size(0) - pos_count
pos_weight = (neg_count / (pos_count + 1e-6)).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'per_label_acc': []}
best_val_loss    = float('inf')
patience_counter = 0

print(f"\n{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val Acc':>8} | {'Time':>6}")
print('-' * 55)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # Train
    model.train()
    tl = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        tl += loss.item() * imgs.size(0)
    train_loss = tl / len(train_loader.dataset)

    # Validate
    model.eval()
    vl, preds_all, lbls_all = 0.0, [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            vl += criterion(logits, labels).item() * imgs.size(0)
            preds_all.append((torch.sigmoid(logits) >= 0.5).float().cpu())
            lbls_all.append(labels.cpu())
    val_loss = vl / len(val_loader.dataset)
    p = torch.cat(preds_all).numpy()
    l = torch.cat(lbls_all).numpy()
    val_acc       = (p == l).mean()
    per_label_acc = (p == l).mean(axis=0)

    scheduler.step(val_loss)
    elapsed = time.time() - t0

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(float(val_acc))
    history['per_label_acc'].append(per_label_acc.tolist())

    print(f'{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | {val_acc:>8.4f} | {elapsed:>5.1f}s')

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), BASE / 'best_model.pth')
        print(f'         → best model 저장 (val_loss={best_val_loss:.4f})')
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}')
        break

with open(BASE / 'history.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\n학습 완료! best val_loss = {best_val_loss:.4f}')

## 셀 5. 시각화 생성

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import f1_score, accuracy_score
import numpy as np, json, torch
from pathlib import Path
from model import GraphoVisionCNN
from data_pipeline import get_dataloaders

BASE    = Path('.')
DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
TRAIT_NAMES = [
    'Emotional\nStability', 'Social\nBehavior', 'Mental\nEnergy', 'Willpower',
    'Imagination', 'Fear /\nAnxiety', 'Introversion\n/Extroversion', 'Sensitivity'
]

# ── ① 학습 곡선 ──────────────────────────────
with open(BASE / 'history.json') as f:
    history = json.load(f)

epochs = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('GraphoVision — Training Curves', fontsize=14, fontweight='bold')
axes[0].plot(epochs, history['train_loss'], label='Train Loss', color='#2196F3')
axes[0].plot(epochs, history['val_loss'],   label='Val Loss',   color='#F44336')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(epochs, history['val_acc'], label='Val Accuracy', color='#4CAF50')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy'); axes[1].set_ylim(0, 1)
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(BASE / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print('저장: training_curves.png')

# ── ② 모델 로드 ───────────────────────────────
model = GraphoVisionCNN().to(DEVICE)
model.load_state_dict(torch.load(BASE / 'best_model.pth', map_location=DEVICE))
model.eval()

_, _, test_loader = get_dataloaders(
    str(BASE/'lines'), str(BASE/'xml'), str(BASE/'label_list.txt'), batch_size=64, num_workers=2
)

# ── ③ 지표별 Accuracy / F1 ───────────────────
preds_all, lbls_all = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        logits = model(imgs.to(DEVICE))
        preds_all.append((torch.sigmoid(logits) >= 0.5).float().cpu())
        lbls_all.append(labels)
P = torch.cat(preds_all).numpy()
L = torch.cat(lbls_all).numpy()

short_names = ['Emot.', 'Social', 'Energy', 'Will', 'Imag.', 'Fear', 'Intro.', 'Sensit.']
accs, f1s = [], []
print(f"\n{'Label':<22} {'Accuracy':>9} {'F1-Score':>9}")
print('-' * 42)
for i in range(8):
    acc = accuracy_score(L[:, i], P[:, i])
    f1  = f1_score(L[:, i], P[:, i], zero_division=0)
    accs.append(acc); f1s.append(f1)
    print(f'  label_{i} ({short_names[i]:<8})  {acc:>8.4f}   {f1:>8.4f}')
print(f"  {'Mean':<20}  {np.mean(accs):>8.4f}   {np.mean(f1s):>8.4f}")

x = np.arange(8)
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - 0.175, accs, 0.35, label='Accuracy', color='#42A5F5', alpha=0.85)
ax.bar(x + 0.175, f1s,  0.35, label='F1-Score',  color='#EF5350', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(short_names, fontsize=10)
ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
ax.set_title('GraphoVision — Per-Label Accuracy & F1-Score (Test Set)', fontsize=13)
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(BASE / 'per_label_metrics.png', dpi=150, bbox_inches='tight')
plt.close()
print('저장: per_label_metrics.png')

# ── ④ 레이더 차트 (4 샘플 배치) ──────────────
imgs_b, labels_b = next(iter(test_loader))
with torch.no_grad():
    probs_b = torch.sigmoid(model(imgs_b.to(DEVICE))).cpu().numpy()
labels_b = labels_b.numpy()

N = 8
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist() + [0]

fig, axes = plt.subplots(2, 2, figsize=(14, 12), subplot_kw={'polar': True})
fig.suptitle('GraphoVision — Ground Truth vs AI Prediction', fontsize=15, fontweight='bold')
for i, ax in enumerate(axes.flatten()):
    gt   = labels_b[i].tolist() + [labels_b[i][0]]
    pred = probs_b[i].tolist()  + [probs_b[i][0]]
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['0.25','0.5','0.75','1.0'], fontsize=7, color='gray')
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(TRAIT_NAMES, fontsize=9, fontweight='bold')
    ax.plot(angles, gt,   'o-',  lw=2.5, color='#1565C0', label='Ground Truth')
    ax.fill(angles, gt,   alpha=0.15, color='#1565C0')
    ax.plot(angles, pred, 's--', lw=2.5, color='#C62828', label='AI Prediction')
    ax.fill(angles, pred, alpha=0.10, color='#C62828')
    ax.set_title(f'Sample {i+1}', fontsize=11, pad=15)

gt_p   = mpatches.Patch(color='#1565C0', alpha=0.7, label='Ground Truth')
pred_p = mpatches.Patch(color='#C62828', alpha=0.7, label='AI Prediction')
fig.legend(handles=[gt_p, pred_p], loc='lower center', ncol=2, fontsize=12, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
plt.savefig(BASE / 'radar_chart_batch.png', dpi=300, bbox_inches='tight')
plt.close()
print('저장: radar_chart_batch.png')

## 셀 6. 결과 이미지 확인

In [ ]:
from IPython.display import Image, display
from pathlib import Path

for fname in ['training_curves.png', 'per_label_metrics.png', 'radar_chart_batch.png']:
    p = Path(fname)
    if p.exists():
        print(f'=== {fname} ===')
        display(Image(filename=str(p)))
    else:
        print(f'[없음] {fname}')

## 셀 7. 결과 파일 다운로드

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

# PNG 파일 + 모델 + 히스토리를 ZIP으로 묶어 다운로드
targets = [
    'training_curves.png',
    'per_label_metrics.png',
    'radar_chart_batch.png',
    'best_model.pth',
    'history.json',
]

with zipfile.ZipFile('graphovision_results.zip', 'w') as zf:
    for fname in targets:
        p = Path(fname)
        if p.exists():
            zf.write(p)
            print(f'  포함: {fname} ({p.stat().st_size // 1024} KB)')
        else:
            print(f'  [건너뜀] {fname}')

files.download('graphovision_results.zip')
print('\nZIP 다운로드 시작!')